# Species Observation Map Analysis with FinBIF

Analyze geographic patterns of species observations using the Finnish Biodiversity Information Facility (FinBIF) R package.

## Overview

This notebook retrieves biodiversity occurrence records directly from FinBIF using the `finbif` R package and analyzes observation patterns for the most frequently observed species. Key analyses include:

- Query occurrence records by taxonomic group or geographic region
- Identify the most frequently observed species in the dataset
- Visualize geographic distribution of observations

**Visualizations:**
- Interactive map displaying observation locations colored by species
- Summary statistics tables

## Step 1: Install and Load Required Packages

In [ ]:
# Install packages if not already installed
required_packages <- c('finbif', 'dplyr', 'ggplot2', 'lubridate', 'leaflet', 'sf', 'ggspatial')

for (pkg in required_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg)
    }
  }

# Load libraries
library(finbif)
library(dplyr)
library(ggplot2)
library(lubridate)
library(leaflet)
library(sf)
library(ggspatial)
cat('\nPackages loaded successfully!')

## Step 2: Configure Query Parameters

In [ ]:
# Set query parameters
# You can modify these to filter by specific criteria
query_group <- c("Cygnus cygnus", "Anser anser", "Branta canadensis")
top_n_species <- 3     # Number of top species to analyze
min_year <- 2010       # Only include observations from this year onwards
limit_records <- 10000  # Limit records to retrieve

## Step 3: Getting a FinBIF access token

To use the FinBIF API you must first request and set a personal access token. You can request an API token to be sent to your email address with the function finbif_get_token().

In [ ]:
finbif_request_token("your@email.com") # Replace with your actual email

Sys.setenv(
  FINBIF_ACCESS_TOKEN = "your-access-token"  # Replace with your actual access token
)

Copy the access token that was sent to your email and set it as the environment variable FINBIF_ACCESS_TOKEN either for the current session, or by adding it to a Renviron startup file (see here for details).

## Step 4: Retrieve Data from FinBIF

In [ ]:
cat('\nFetching occurrence records from FinBIF...\n')

# Filter the records
finbif_records = finbif_occurrence(
  species = query_group,
  filter = list(coordinates_uncertainty_max  = 100),
  n = limit_records
)

cat('\nFirst few records:')
print(head(finbif_records, 5))

## Step 5: Data Processing and Cleaning

In [ ]:
# Convert to data frame and process dates
records_df <- as_tibble(finbif_records) %>%
  rename(
    species = scientific_name,
    latitude = lat_wgs84,
    longitude = lon_wgs84,
    date = date_time
  ) %>%
  mutate(
    date = as.Date(date),
    year = year(date)
  ) %>%
  filter(
    !is.na(species),
    !is.na(latitude),
    !is.na(longitude),
    !is.na(year),
    year >= min_year
  )

cat('\nProcessed data summary:')
cat('\nTotal records after cleaning:', nrow(records_df))
cat('\nDate range:', min(records_df$year), 'to', max(records_df$year))
cat('\nNumber of unique species:', n_distinct(records_df$species))

## Step 6: Identify Top Species

In [ ]:
# Identify top species by observation count
top_species <- records_df %>%
  group_by(species) %>%
  summarise(
    observation_count = n(),
    .groups = 'drop'
  ) %>%
  arrange(desc(observation_count)) %>%
  slice_head(n = top_n_species) %>%
  pull(species)

cat('\nTop', top_n_species, 'most frequently observed species:\n')
print(
  records_df %>%
    group_by(species) %>%
    summarise(count = n(), .groups = 'drop') %>%
    arrange(desc(count)) %>%
    slice_head(n = top_n_species)
)

# Filter data to top species
records_top <- records_df %>%
  filter(species %in% top_species)

cat('\nRecords for top', top_n_species, 'species:', nrow(records_top))

## Step 7: Temporal Trend Analysis

In [ ]:
# Calculate summary statistics
trend_summary <- records_top %>%
  group_by(species) %>%
  summarise(
    total_observations = n(),
    min_year = min(year),
    max_year = max(year),
    avg_per_year = round(n() / n_distinct(year), 2),
    .groups = 'drop'
  ) %>%
  arrange(desc(total_observations))

cat('\nTrend summary by species:\n')
print(trend_summary)

## Step 8: Create Geographic Map

In [ ]:
# Define color palette for species
species_colors <- c(
  '#e41a1c', '#377eb8', '#4daf4a', '#984ea3',
  '#ff7f00', '#a65628', '#f781bf', '#999999'
)

color_mapping <- setNames(
  species_colors[1:length(top_species)],
  top_species
)

# Create simple geographic map
observation_map <- ggplot(records_top, aes(x = longitude, y = latitude, color = species)) +
  geom_point(size = 3, alpha = 0.6) +
  scale_color_manual(values = color_mapping) +
  labs(
    title = paste('Geographic Distribution of Top', top_n_species, 'Species'),
    x = 'Longitude',
    y = 'Latitude',
    color = 'Species'
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(size = 14, face = 'bold'),
    legend.position = 'right'
  )

print(observation_map)

## Step 10: Geographic Distribution Summary

In [ ]:
# Calculate geographic statistics by species
geographic_summary <- records_top %>%
  group_by(species) %>%
  summarise(
    total_observations = n(),
    unique_locations = n_distinct(paste0(round(latitude, 2), round(longitude, 2))),
    min_latitude = round(min(latitude, na.rm = TRUE), 2),
    max_latitude = round(max(latitude, na.rm = TRUE), 2),
    min_longitude = round(min(longitude, na.rm = TRUE), 2),
    max_longitude = round(max(longitude, na.rm = TRUE), 2),
    .groups = 'drop'
  ) %>%
  arrange(desc(total_observations))

cat('\nGeographic Distribution Summary:\n')
print(geographic_summary)

cat('\n\nAnalysis complete! Summary statistics:')
cat('\nTotal observations analyzed:', nrow(records_top))
cat('\nTime span:', min(records_top$year), 'to', max(records_top$year))
cat('\nNumber of species:', n_distinct(records_top$species))